In [2]:
%pip install selenium webdriver-manager

  Using cached selenium-4.41.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata (12 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached trio-0.33.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached cffi-2.0.0-cp313-cp313-win_amd64.whl.metadata (2.6 kB)
  Using cached wsproto-1.3.2-py3-non


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import time
import json
import re  # 用於提取總評論數中的數字
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# 設定目標
HOTEL_NAME = "台北明日酒店" 
HOTEL_URL = "https://www.agoda.com/zh-hk/grand-hotel/reviews/taipei-tw.html?countryId=140&finalPriceView=1&isShowMobileAppPrice=false&cid=-1&numberOfBedrooms=&familyMode=false&adults=1&children=0&rooms=1&maxRooms=0&checkIn=2026-04-13&isCalendarCallout=false&childAges=&numberOfGuest=0&missingChildAges=false&travellerType=0&showReviewSubmissionEntry=false&currencyCode=TWD&isFreeOccSearch=false&flightSearchCriteria=%5bobject%2520Object%5d&tspTypes=16&los=2&searchrequestid=db1d2fc4-14e5-493a-b29b-f0ac853ee866&ds=sGdigSj%2f%2bnt1%2fasJ"
MAX_REVIEWS = 15  # 抓取上限

def get_total_reviews(driver):
    """從頁面中定位並提取總評論數"""
    try:
        # 根據截圖定位：尋找包含「來自AGODA用戶」字樣的元素
        element = driver.find_element(By.XPATH, "//p[contains(text(), '來自AGODA用戶')]")
        text = element.text # 例如: "來自AGODA用戶 (共26,933條)"
        
        # 使用正則表達式提取數字（包含處理千分位逗號）
        numbers = re.findall(r'\d+(?:,\d+)*', text)
        if numbers:
            # 去除逗號並轉為整數
            total = int(numbers[0].replace(',', ''))
            return total
    except Exception as e:
        print(f"⚠️ 無法取得總評論數，將使用預設上限。錯誤: {e}")
    return None

def run_agoda_spider(url, max_target):
    options = webdriver.ChromeOptions()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    all_results = []
    seen_ids = set()

    try:
        driver.get(url)
        wait = WebDriverWait(driver, 20)
        time.sleep(3) # 等待頁面基礎架構加載

        # --- 新增：獲取總數並動態修正目標 ---
        total_available = get_total_reviews(driver)
        if total_available:
            print(f"📊 偵測到飯店實際評論總數: {total_available}")
            target_count = min(max_target, total_available)
        else:
            target_count = max_target
        
        print(f"🎯 最終抓取目標設定為: {target_count} 則\n")

        while len(all_results) < target_count:
            # 顯示進度百分比
            percent = (len(all_results) / target_count) * 100
            print(f"📜 進度: {percent:.1f}% | 已抓取: {len(all_results)}/{target_count}...")
            
            # 深度滾動確保元素加載
            for _ in range(5):
                driver.execute_script("window.scrollBy(0, 1000);")
                time.sleep(0.5)
            
            card_selector = '[data-element-name="review-comment"]'
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, card_selector)))
            cards = driver.find_elements(By.CSS_SELECTOR, card_selector)

            for card in cards:
                if len(all_results) >= target_count: break
                
                try:
                    r_id = card.get_attribute("data-review-id")
                    if not r_id or r_id in seen_ids: continue
                    
                    score_el = card.find_elements(By.CSS_SELECTOR, ".Review-comment-leftScore")
                    if not score_el: continue

                    # 展開「閱讀更多」
                    try:
                        read_more_btn = card.find_elements(By.CSS_SELECTOR, '[data-element-name="review-read-more-button"]')
                        if read_more_btn:
                            driver.execute_script("arguments[0].click();", read_more_btn[0])
                            time.sleep(0.3)
                    except: pass

                    # 提取基本資料
                    score = score_el[0].text
                    content = card.find_element(By.CSS_SELECTOR, ".Review-comment-bodyText").text.strip()
                    
                    # 處理日期
                    comment_date = "未知日期"
                    all_spans = card.find_elements(By.TAG_NAME, "span")
                    for s in all_spans:
                        txt = s.text
                        if "202" in txt and ("評價" in txt or "發表" in txt):
                            comment_date = txt
                            break

                    # 提取「房型」與「旅遊類型」
                    room_type = ""
                    travel_type = ""
                    try:
                        detail_info = card.find_elements(By.CSS_SELECTOR, '.Review-comment-reviewer-subInfo .Review-comment-reviewer-subInfo-item')
                        if len(detail_info) >= 1: travel_type = detail_info[0].text
                        if len(detail_info) >= 2: room_type = detail_info[1].text
                    except: pass

                    # 抓取照片數量
                    photo_count = 0
                    try:
                        photo_buttons = card.find_elements(By.CSS_SELECTOR, 'button[data-element-name="review-comment-ugc-thumbnail"]')
                        photo_count = len(photo_buttons)
                    except: pass

                    # 封裝 JSON
                    review_item = {
                        "飯店名稱": HOTEL_NAME,
                        "評論ID": r_id,
                        "評分": score,
                        "評論內容": content,
                        "評論日期": comment_date,
                        "旅遊類型": travel_type,
                        "房型": room_type,
                        "照片數量": photo_count
                    }

                    seen_ids.add(r_id)
                    all_results.append(review_item)
                    print(f"✅ ID: {r_id} | ⭐: {score} | 📷: {photo_count}")

                except Exception:
                    continue

            if len(all_results) >= target_count: break

            # 翻頁邏輯：點擊「顯示更多評價」
            try:
                load_more_btn = driver.find_elements(By.CSS_SELECTOR, '.Review-paginator-button')
                
                if load_more_btn and load_more_btn[0].is_displayed():
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", load_more_btn[0])
                    time.sleep(1)
                    driver.execute_script("arguments[0].click();", load_more_btn[0])
                    # 等待新評論加載
                    time.sleep(3) 
                else:
                    print("✨ 已無更多評論可載入。")
                    break
            except Exception as e:
                print(f"🛑 載入按鈕處理異常: {e}")
                break

        # --- 導出 JSON 檔案 ---
        file_name = "agoda_reviews.json"
        with open(file_name, "w", encoding="utf-8") as f:
            json.dump(all_results, f, ensure_ascii=False, indent=2)
        
        print("\n" + "★" * 20 + " 最終報告 " + "★" * 20)
        print(f"📊 飯店總評論數: {total_available if total_available else '未知'}")
        print(f"✅ 成功抓取總數: {len(all_results)}")
        print(f"📂 數據已存入: {file_name}")

    except Exception as e:
        print(f"❌ 執行出錯: {e}")
    finally:
        if 'driver' in locals():
            input("\n確認完畢請按 Enter 關閉瀏覽器...")
            driver.quit()

if __name__ == "__main__":
    run_agoda_spider(HOTEL_URL, MAX_REVIEWS)

📊 偵測到飯店實際評論總數: 12174
🎯 最終抓取目標設定為: 15 則

📜 進度: 0.0% | 已抓取: 0/15...
✅ ID: 1070112281 | ⭐: 8.8 | 📷: 0
✅ ID: 1069152642 | ⭐: 10.0 | 📷: 0
✅ ID: 1064938054 | ⭐: 8.8 | 📷: 0
✅ ID: 1064935880 | ⭐: 8.4 | 📷: 0
✅ ID: 1064571450 | ⭐: 10.0 | 📷: 0
✅ ID: 1064718136 | ⭐: 9.6 | 📷: 1
✅ ID: 1063911456 | ⭐: 9.2 | 📷: 0
✅ ID: 1062884513 | ⭐: 7.2 | 📷: 0
✅ ID: 1062851017 | ⭐: 9.6 | 📷: 0
✅ ID: 1062606567 | ⭐: 9.6 | 📷: 8
✅ ID: 1061609512 | ⭐: 2.0 | 📷: 0
✅ ID: 1060373562 | ⭐: 9.6 | 📷: 0
✅ ID: 1060107899 | ⭐: 9.2 | 📷: 0
✅ ID: 1059506510 | ⭐: 9.6 | 📷: 0
✅ ID: 1059066000 | ⭐: 9.2 | 📷: 0

★★★★★★★★★★★★★★★★★★★★ 最終報告 ★★★★★★★★★★★★★★★★★★★★
📊 飯店總評論數: 12174
✅ 成功抓取總數: 15
📂 數據已存入: agoda_reviews.json
